# MERU UNIVERSITY OF SCIENCE AND TECHNOLOGY
## School of Computing and Informatics | Department of Computer Science
### CDS 3301: Machine Learning — Group 2 Mini-Project
**Project Theme:** Diabetes Risk Screening (Supervised Classification)
**Lecturer:** Dr. Amos Chege Kirongo
**Group Members:**
1. Ordit Kagwira (CT202/114188/23) — Coordinator & Report Lead
2. Beauttah Kipruttoh (CT202/113636/23) — Data Acquisition & Cleaning
3. Atitwa Samuel Makokha (CT202/113631/23) — EDA & Visualisations
4. Kiplangat Dancun (CT202/113622/23) — Model Implementation & Prototype
5. Wamukota Kevin (CT202/106143/21) — Hyperparameter Tuning & Evaluation


## 1. Environment Setup & Data Loading
We load the genuine Pima Indians Diabetes Database and inspect raw features.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             fbeta_score, roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix)

df = pd.read_csv("../dataset/diabetes.csv")
print(f"Dataset shape: {df.shape}")
display(df.head())


## 2. Biological Zero-Value Audit & Preprocessing
Physiological values like Glucose, Blood Pressure, Skin Thickness, Insulin, and BMI cannot be zero in living humans.
We replace them with NaN and impute using median strategy fitted exclusively on training data to prevent data leakage.

In [2]:
bio_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("Biological zero counts:")
for col in bio_cols:
    z_cnt = (df[col] == 0).sum()
    print(f"- {col}: {z_cnt} zeros ({z_cnt/len(df)*100:.2f}%)")

# Stratified Train-Test split
X = df.drop(columns=["Outcome"])
y = df["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Replace zeros with NaN on X_train and X_test
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()
for col in bio_cols:
    X_train_clean[col] = X_train_clean[col].replace(0, np.nan)
    X_test_clean[col] = X_test_clean[col].replace(0, np.nan)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

# Fit only on train!
X_train_proc = scaler.fit_transform(imputer.fit_transform(X_train_clean))
X_test_proc = scaler.transform(imputer.transform(X_test_clean))
print(f"Training set shape: {X_train_proc.shape}, Test set shape: {X_test_proc.shape}")


## 3. Training & 5-Fold Stratified Cross-Validation
We train all six algorithms: Dummy Baseline, Logistic Regression, KNN, Decision Tree, Random Forest, and SVM.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "dummy": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), {"C": [0.1, 1.0, 5.0]}, cv=cv, scoring="roc_auc"),
    "knn": GridSearchCV(KNeighborsClassifier(), {"n_neighbors": [5, 9, 13]}, cv=cv, scoring="roc_auc"),
    "decision_tree": GridSearchCV(DecisionTreeClassifier(random_state=42), {"max_depth": [3, 4, 6], "min_samples_leaf": [5, 10]}, cv=cv, scoring="roc_auc"),
    "random_forest": GridSearchCV(RandomForestClassifier(random_state=42), {"n_estimators": [100, 150], "max_depth": [4, 6], "class_weight": ["balanced"]}, cv=cv, scoring="roc_auc"),
    "svm": GridSearchCV(SVC(probability=True, random_state=42), {"C": [1.0, 2.0], "kernel": ["rbf"]}, cv=cv, scoring="roc_auc")
}

fitted_models = {}
for name, m in models.items():
    m.fit(X_train_proc, y_train)
    best = m.best_estimator_ if hasattr(m, "best_estimator_") else m
    fitted_models[name] = best
    print(f"Trained {name}")


## 4. Test Set Evaluation & Threshold Selection (tau = 0.35)
Why Recall is vital: In screening, False Negatives leave diabetic patients undiagnosed, leading to severe organ failure.
Lowering threshold to tau = 0.35 boosts recall to over 90% in Random Forest.

In [4]:
for name, model in fitted_models.items():
    proba = model.predict_proba(X_test_proc)[:, 1] if hasattr(model, "predict_proba") else model.predict(X_test_proc)
    pred_05 = (proba >= 0.50).astype(int)
    pred_035 = (proba >= 0.35).astype(int)
    rec_05 = recall_score(y_test, pred_05, zero_division=0)
    rec_035 = recall_score(y_test, pred_035, zero_division=0)
    auc_score = roc_auc_score(y_test, proba) if hasattr(model, "predict_proba") else 0.5
    print(f"{name:20s} | AUC: {auc_score:.3f} | Recall (tau=0.50): {rec_05:.1%} | Recall (tau=0.35): {rec_035:.1%}")
